In [ ]:
!pip install -q langgraph-supervisor pydantic langchain==0.3.27 langchain-openai==0.3.33 langchain-community==0.3.24 --q

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["OPENAI_BASE_URL"] = "https://openai.vocareum.com/v1"
os.environ["LANGCHAIN_API_KEY"] = ""
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Multi-Agent Supervisor Demo"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor

In [ ]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
MATH_PROMPT ="You are an expert Math Assistant specializing in calculations, algebra, calculus, statistics, and step-by-step problem solving."
math_agent = create_react_agent(
    model=model,
    tools=[],
    name="math_agent",
    prompt=MATH_PROMPT
)

In [ ]:
CS_PROMPT ="You are an expert Computer Science Assistant specializing in programming, algorithms, data structures, software engineering, and code debugging."
cs_agent = create_react_agent(
    model=model,
    tools=[],
    name="cs_agent",
    prompt=CS_PROMPT
)

In [ ]:
LANG_PROMPT = "You are an expert Language and Translation Assistant specializing in translating text between languages, explaining grammatical nuances, and linguistics."

# Agent creation
language_agent = create_react_agent(
    model=model,
    tools=[],
    name="language_agent",
    prompt=LANG_PROMPT
)

In [ ]:
ENG_PROMPT = "You are an expert English & Writing Assistant specializing in literature analysis, essay writing, editing, and grammar feedback."
english_agent = create_react_agent(
    model=model,
    tools=[],
    name="english_agent",
    prompt=ENG_PROMPT
)

In [ ]:
GEN_PROMPT = "You are a General Knowledge Assistant specializing in history, science, geography, art, and general trivia queries."
general_agent = create_react_agent(
    model=model,
    tools=[],
    name="general_agent",
    prompt=GEN_PROMPT
)

In [ ]:
SUPERVISOR_PROMPT = """You are TeachAssist, a primary educational supervisor agent.
  Your job is to analyze incoming student queries and route them to the most suitable subject specialist:
- math_agent: for math, algebra, calculus, calculations, and statistics.
- cs_agent: for programming, computer science, code snippets, and algorithms.
- language_agent: for multi-language translations and language learning.
- english_agent: for essay writing, literature, grammar, and essay editing.
- general_agent: for history, social studies, general science, and general knowledge.

ROUTING RULES:
1. Route the student's question to the appropriate specialist agent.
2. Once the specialist agent answers the question, present their response to the user and IMMEDIATELY call the completion tool to stop.
Do not route back to the specialist or ask follow-up questions.
"""
workflow = create_supervisor(
    agents=[math_agent, cs_agent, language_agent, english_agent, general_agent],
    model=model,
    prompt=SUPERVISOR_PROMPT,
    add_handoff_back_messages=False,
    output_mode="last_message",  # Direct output from sub-agents without secondary summarization
)

In [ ]:
app = workflow.compile()

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
def ask_teacher(query: str):
    print(f"\n{'='*60}\nStudent: {query}\n{'='*60}\n")
    print("TeachAssist: ", end="", flush=True)

    # Use app.stream to capture real-time tokens
    for chunk, metadata in app.stream(
        {"messages": [{"role": "user", "content": query}]},
        stream_mode="messages"
    ):
        content = chunk.content
        node_name = metadata.get("langgraph_node", "")

        # 1. Ignore empty chunks or supervisor control messages
        if not content or node_name == "supervisor":
            continue

        # 2. Filter out automatic handoff system notifications
        if isinstance(content, str) and (
            content.startswith("Transferring back")
            or "Successfully transferred" in content
        ):
            continue

        # 3. Print only the actual content from the specialist
        print(content, end="", flush=True)

    print("\n")

In [ ]:
ask_teacher("Can you explain what is an algorithm in less than 50 words?")

In [ ]:
ask_teacher("Translate 'Where is the library?' into Spanish.")

In [ ]:
ask_teacher("Can you tell me what is 5+2")